# 03. VWAP 타겟 설계 판단 노트

`01_ets_eda.ipynb`의 기초 same-vintage 타겟 EDA와 `02_vintage_rule_eda.ipynb`의 roll rule 진단 결과를 합쳐서, 다음 단계에서 사용할 타겟 후보와 표본 정책을 정리한다.

중요한 구분은 다음과 같다.

- `01_ets_eda.ipynb`: KRX ETS 원천 구조, 빈티지별 거래 구조, same-vintage 30d/60d coverage·분포·극단값을 넓게 확인한다.
- `02_vintage_rule_eda.ipynb`: 인접 빈티지를 이어 붙이는 `volume_confirmed_roll` 후보가 안정적인지 확인한다.
- 이 노트북: 01과 02를 반복하지 않고, 모델링 전 타겟 후보·horizon·거래량 필터·저장 스키마를 결정하기 위한 요약 판단표를 만든다.

In [24]:
# 1. 환경 설정
from pathlib import Path
import os
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

if Path('/mnt/hgfs/Windows/Climate').exists():
    PROJECT_ROOT = Path('/mnt/hgfs/Windows/Climate')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

In [28]:
# 2. DB 연결 및 KRX ETS 원천 데이터 로드
load_dotenv(PROJECT_ROOT / '.env')
required_keys = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
missing = [key for key in required_keys if not os.getenv(key)]
if missing:
    raise RuntimeError(f'필수 DB 환경변수가 없습니다: {missing}')

user = os.getenv('DB_USER')
password = quote_plus(os.getenv('DB_PASSWORD'))
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db = os.getenv('DB_NAME')
engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}')

query = '''
select
    period,
    trd_dd,
    isu_code,
    clsprc,
    acc_trdvol,
    acc_trdval,
    vwap
from raw.krx_ets_daily
order by trd_dd, isu_code
'''

ets = pd.read_sql(query, engine, parse_dates=['trd_dd'])
ets['isu_year'] = ets['isu_code'].str.extract(r'(\d{2})$')[0].astype('Int64') + 2000
ets['is_trade_day'] = (ets['acc_trdvol'] > 0) & (ets['vwap'] > 0)
ets_trade = ets[ets['is_trade_day']].copy().sort_values(['isu_code', 'trd_dd']).reset_index(drop=True)

source_summary = pd.DataFrame({
    'item': [
        'raw_rows',
        'trade_rows',
        'date_min',
        'date_max',
        'traded_vintage_count',
        'traded_vintages',
    ],
    'value': [
        len(ets),
        len(ets_trade),
        ets['trd_dd'].min().date(),
        ets['trd_dd'].max().date(),
        ets_trade['isu_code'].nunique(),
        ', '.join(sorted(ets_trade['isu_code'].unique())),
    ],
})
source_summary

,item,value
0,raw_rows,8213
1,trade_rows,2385
2,date_min,2015-01-12
3,date_max,2026-04-23
4,traded_vintage_count,11
5,traded_vintages,"KAU15, KAU16, KAU17, KAU18, KAU19, KAU20, KAU2..."


In [25]:
# 3. same-vintage 후보 핵심 지표만 재계산
def add_same_vintage_targets(df, horizons=(30, 60)):
    out = df.sort_values(['isu_code', 'trd_dd']).copy().reset_index(drop=True)
    for horizon in horizons:
        out[f'future_date_{horizon}d'] = pd.NaT
        out[f'future_vwap_{horizon}d'] = np.nan
        out[f'future_volume_{horizon}d'] = np.nan
        out[f'actual_horizon_days_{horizon}d'] = np.nan
        out[f'target_vwap_logret_{horizon}d'] = np.nan

    for isu_code, g in out.groupby('isu_code', sort=False):
        idx = g.index.to_numpy()
        dates = g['trd_dd'].to_numpy(dtype='datetime64[ns]')
        vwap = g['vwap'].to_numpy(dtype=float)
        volume = g['acc_trdvol'].to_numpy(dtype=float)
        for horizon in horizons:
            target_dates = dates + np.timedelta64(horizon, 'D')
            future_pos = np.searchsorted(dates, target_dates, side='left')
            valid = future_pos < len(g)
            if not valid.any():
                continue
            base_idx = idx[valid]
            pos = future_pos[valid]
            future_dates = pd.to_datetime(dates[pos])
            actual_horizon_days = (
                future_dates.to_numpy(dtype='datetime64[ns]') - dates[valid]
            ) / np.timedelta64(1, 'D')
            out.loc[base_idx, f'future_date_{horizon}d'] = future_dates
            out.loc[base_idx, f'future_vwap_{horizon}d'] = vwap[pos]
            out.loc[base_idx, f'future_volume_{horizon}d'] = volume[pos]
            out.loc[base_idx, f'actual_horizon_days_{horizon}d'] = actual_horizon_days
            out.loc[base_idx, f'target_vwap_logret_{horizon}d'] = np.log(vwap[pos] / vwap[valid])
    return out

horizons = [30, 60]
target_panel = add_same_vintage_targets(ets_trade, horizons=horizons)

same_vintage_rows = []
for horizon in horizons:
    target_col = f'target_vwap_logret_{horizon}d'
    actual_col = f'actual_horizon_days_{horizon}d'
    future_vol_col = f'future_volume_{horizon}d'
    valid = target_panel[target_col].notna()
    s = target_panel.loc[valid, target_col]
    actual = target_panel.loc[valid, actual_col]
    same_vintage_rows.append({
        'candidate': f'same_vintage_{horizon}d',
        'target_count': int(valid.sum()),
        'coverage': valid.mean(),
        'actual_horizon_median': actual.median(),
        'actual_horizon_p95': actual.quantile(0.95),
        'actual_horizon_max': actual.max(),
        'return_mean': s.mean(),
        'return_std': s.std(),
        'return_median': s.median(),
        'return_min': s.min(),
        'return_max': s.max(),
        'positive_rate': (s > 0).mean(),
        'keep_rate_min_volume_1000': (
            valid & (target_panel['acc_trdvol'] >= 1_000) & (target_panel[future_vol_col] >= 1_000)
        ).sum() / valid.sum(),
        'keep_rate_min_volume_10000': (
            valid & (target_panel['acc_trdvol'] >= 10_000) & (target_panel[future_vol_col] >= 10_000)
        ).sum() / valid.sum(),
    })

same_vintage_decision_summary = pd.DataFrame(same_vintage_rows)
same_vintage_decision_summary_display = same_vintage_decision_summary.rename(columns={
    'candidate': '타겟 후보',
    'target_count': '타겟 수',
    'coverage': 'coverage',
    'actual_horizon_median': '실제 경과일 중앙값',
    'actual_horizon_p95': '실제 경과일 95% 분위',
    'actual_horizon_max': '실제 경과일 최대값',
    'return_mean': '수익률 평균',
    'return_std': '수익률 표준편차',
    'return_median': '수익률 중앙값',
    'return_min': '수익률 최소값',
    'return_max': '수익률 최대값',
    'positive_rate': '상승 비율',
    'keep_rate_min_volume_1000': '거래량≥1,000 유지율',
    'keep_rate_min_volume_10000': '거래량≥10,000 유지율',
})

same_vintage_decision_summary_display.style.format({
    '타겟 수': '{:,.0f}',
    'coverage': '{:.1%}',
    '실제 경과일 중앙값': '{:.1f}',
    '실제 경과일 95% 분위': '{:.1f}',
    '실제 경과일 최대값': '{:.1f}',
    '수익률 평균': '{:.4f}',
    '수익률 표준편차': '{:.4f}',
    '수익률 중앙값': '{:.4f}',
    '수익률 최소값': '{:.4f}',
    '수익률 최대값': '{:.4f}',
    '상승 비율': '{:.1%}',
    '거래량≥1,000 유지율': '{:.1%}',
    '거래량≥10,000 유지율': '{:.1%}',
})

,타겟 후보,타겟 수,coverage,실제 경과일 중앙값,실제 경과일 95% 분위,실제 경과일 최대값,수익률 평균,수익률 표준편차,수익률 중앙값,수익률 최소값,수익률 최대값,상승 비율,"거래량≥1,000 유지율","거래량≥10,000 유지율"
0,same_vintage_30d,"2,168",90.9%,30.0,35.0,268.0,-0.0098,0.1416,0.0036,-0.7342,0.7438,52.4%,97.0%,68.5%
1,same_vintage_60d,"1,962",82.3%,60.0,64.0,268.0,-0.0321,0.1867,-0.0031,-0.6461,0.8100,47.7%,97.3%,68.3%


In [ ]:
# 4. 극단 수익률이 저유동성 표본에 얼마나 걸리는지 확인
extreme_policy_rows = []
for horizon in horizons:
    target_col = f'target_vwap_logret_{horizon}d'
    future_vol_col = f'future_volume_{horizon}d'
    base = target_panel[target_panel[target_col].notna()].copy()
    abs_threshold = base[target_col].abs().quantile(0.99)
    extreme = base[base[target_col].abs() >= abs_threshold].copy()
    for volume_threshold in [1_000, 10_000]:
        low_liq = (extreme['acc_trdvol'] < volume_threshold) | (extreme[future_vol_col] < volume_threshold)
        all_low_liq = (base['acc_trdvol'] < volume_threshold) | (base[future_vol_col] < volume_threshold)
        extreme_policy_rows.append({
            'horizon_days': horizon,
            'abs_return_p99_threshold': abs_threshold,
            'extreme_count': len(extreme),
            'volume_threshold': volume_threshold,
            'extreme_low_liquidity_count': int(low_liq.sum()),
            'extreme_low_liquidity_rate': low_liq.mean(),
            'all_low_liquidity_rate': all_low_liq.mean(),
        })

extreme_liquidity_check = pd.DataFrame(extreme_policy_rows)
extreme_liquidity_check_display = extreme_liquidity_check.rename(columns={
    'horizon_days': '목표 기간',
    'abs_return_p99_threshold': '|수익률| 99% 기준',
    'extreme_count': '극단 표본 수',
    'volume_threshold': '저유동성 기준',
    'extreme_low_liquidity_count': '극단 중 저유동성 수',
    'extreme_low_liquidity_rate': '극단 중 저유동성 비율',
    'all_low_liquidity_rate': '전체 중 저유동성 비율',
})

extreme_liquidity_check_display.style.format({
    '|수익률| 99% 기준': '{:.4f}',
    '극단 표본 수': '{:,.0f}',
    '저유동성 기준': '{:,.0f}',
    '극단 중 저유동성 수': '{:,.0f}',
    '극단 중 저유동성 비율': '{:.1%}',
    '전체 중 저유동성 비율': '{:.1%}',
})

,목표 기간,|수익률| 99% 기준,극단 표본 수,저유동성 기준,극단 중 저유동성 수,극단 중 저유동성 비율,전체 중 저유동성 비율
0,30,0.5144,22,"1,000",6,27.3%,3.0%
1,30,0.5144,22,"10,000",11,50.0%,31.5%
2,60,0.5695,20,"1,000",1,5.0%,2.7%
3,60,0.5695,20,"10,000",9,45.0%,31.7%


In [ ]:
# 5. volume-confirmed roll 후보 계산
def adjacent_pairs(codes):
    years = pd.Series(codes).str.extract(r'(\d{2})$')[0].astype(int) + 2000
    pairs = pd.DataFrame({'isu_code': list(codes), 'isu_year': years}).sort_values('isu_year')
    rows = []
    for i in range(len(pairs) - 1):
        cur = pairs.iloc[i]
        nxt = pairs.iloc[i + 1]
        if int(nxt['isu_year']) == int(cur['isu_year']) + 1:
            rows.append((cur['isu_code'], nxt['isu_code']))
    return rows


def build_pair_overlap(df, current_code, next_code):
    cur = df[df['isu_code'] == current_code][['trd_dd', 'acc_trdvol', 'vwap']].rename(columns={
        'acc_trdvol': 'current_volume',
        'vwap': 'current_vwap',
    })
    nxt = df[df['isu_code'] == next_code][['trd_dd', 'acc_trdvol', 'vwap']].rename(columns={
        'acc_trdvol': 'next_volume',
        'vwap': 'next_vwap',
    })
    overlap = cur.merge(nxt, on='trd_dd', how='inner').sort_values('trd_dd').reset_index(drop=True)
    overlap['current_code'] = current_code
    overlap['next_code'] = next_code
    overlap['pair'] = f'{current_code}->{next_code}'
    return overlap


def find_volume_confirmed_roll(overlap, rolling_window=20, confirm_days=5, min_next_volume=1_000):
    x = overlap.sort_values('trd_dd').copy()
    x['current_rolling_volume'] = x['current_volume'].rolling(rolling_window, min_periods=1).sum()
    x['next_rolling_volume'] = x['next_volume'].rolling(rolling_window, min_periods=1).sum()
    x['next_gt_current'] = x['next_rolling_volume'] > x['current_rolling_volume']
    x['next_volume_ok'] = x['next_volume'] >= min_next_volume
    x['candidate_flag'] = x['next_gt_current'] & x['next_volume_ok']
    x['roll_signal_run'] = (
        x['candidate_flag']
        .astype(int)
        .groupby((~x['candidate_flag']).cumsum())
        .cumsum()
    )
    confirmed = x[x['roll_signal_run'] >= confirm_days].copy()
    if confirmed.empty:
        return None
    row = confirmed.iloc[0]
    post = x[x['trd_dd'] >= row['trd_dd']].copy()
    next_win_rate_after = post['next_gt_current'].mean() if len(post) else np.nan
    return {
        'pair': row['pair'],
        'roll_candidate_date': row['trd_dd'],
        'current_rolling_volume': row['current_rolling_volume'],
        'next_rolling_volume': row['next_rolling_volume'],
        'current_volume': row['current_volume'],
        'next_volume': row['next_volume'],
        'price_ratio_next_current': row['next_vwap'] / row['current_vwap'],
        'log_price_gap': np.log(row['next_vwap'] / row['current_vwap']),
        'post_overlap_days': len(post),
        'post_next_win_rate': next_win_rate_after,
    }

pair_rows = []
roll_rows = []
for current_code, next_code in adjacent_pairs(sorted(ets_trade['isu_code'].unique())):
    overlap = build_pair_overlap(ets_trade, current_code, next_code)
    if overlap.empty:
        continue
    pair_rows.append({
        'pair': f'{current_code}->{next_code}',
        'overlap_trade_days': len(overlap),
        'next_daily_volume_win_rate': (overlap['next_volume'] > overlap['current_volume']).mean(),
    })
    candidate = find_volume_confirmed_roll(overlap)
    if candidate is not None:
        roll_rows.append(candidate)

pair_overlap_summary = pd.DataFrame(pair_rows)
roll_candidate_summary = pd.DataFrame(roll_rows)

if roll_candidate_summary.empty:
    roll_candidate_display = pd.DataFrame([{'결과': 'volume_confirmed_w20_c5_v1000 후보 없음'}])
else:
    roll_candidate_display = roll_candidate_summary.rename(columns={
        'pair': 'pair',
        'roll_candidate_date': '전환 후보일',
        'current_rolling_volume': '현재 빈티지 rolling 거래량',
        'next_rolling_volume': '다음 빈티지 rolling 거래량',
        'current_volume': '후보일 현재 빈티지 거래량',
        'next_volume': '후보일 다음 빈티지 거래량',
        'price_ratio_next_current': '후보일 가격비율(next/current)',
        'log_price_gap': '후보일 log price gap',
        'post_overlap_days': '후보일 이후 overlap 일수',
        'post_next_win_rate': '후보일 이후 next rolling 우위 비율',
    })

roll_candidate_display

,pair,전환 후보일,현재 빈티지 rolling 거래량,다음 빈티지 rolling 거래량,후보일 현재 빈티지 거래량,후보일 다음 빈티지 거래량,후보일 가격비율(next/current),후보일 log price gap,후보일 이후 overlap 일수,후보일 이후 next rolling 우위 비율
0,KAU19->KAU20,2020-04-09,157209.0,218588.0,36000,5000,1.02497,0.024663,52,0.192308


In [21]:
# 6. 타겟 후보별 의사결정 scorecard
same_30 = same_vintage_decision_summary.set_index('candidate').loc['same_vintage_30d']
same_60 = same_vintage_decision_summary.set_index('candidate').loc['same_vintage_60d']

if roll_candidate_summary.empty:
    roll_count = 0
    roll_status = '후보 없음'
    roll_limit = 'volume_confirmed_w20_c5_v1000 기준 전환 후보가 없음'
else:
    roll_count = len(roll_candidate_summary)
    post_win = roll_candidate_summary['post_next_win_rate'].iloc[0]
    roll_status = f'후보 {roll_count}개, 후보일 이후 next 우위 {post_win:.1%}'
    roll_limit = '후보가 적고 전환 이후 다음 빈티지 우위 지속성이 약함'

scorecard = pd.DataFrame([
    {
        'target_candidate': 'same_vintage_30d',
        'coverage': same_30['coverage'],
        'interpretability': '높음: 기준일과 미래일의 isu_code가 동일함',
        'horizon_stability': f"중앙값 {same_30['actual_horizon_median']:.0f}일, 95% {same_30['actual_horizon_p95']:.0f}일",
        'liquidity_policy': f"거래량≥1,000 적용 시 {same_30['keep_rate_min_volume_1000']:.1%} 유지",
        'main_risk': '60d보다 정책·저빈도 변수 반영 기간이 짧을 수 있음',
        'decision': '주력 후보',
    },
    {
        'target_candidate': 'same_vintage_60d',
        'coverage': same_60['coverage'],
        'interpretability': '높음: 기준일과 미래일의 isu_code가 동일함',
        'horizon_stability': f"중앙값 {same_60['actual_horizon_median']:.0f}일, 95% {same_60['actual_horizon_p95']:.0f}일",
        'liquidity_policy': f"거래량≥1,000 적용 시 {same_60['keep_rate_min_volume_1000']:.1%} 유지",
        'main_risk': '30d보다 coverage가 낮고 표본 말미 결측이 커짐',
        'decision': '주력 후보',
    },
    {
        'target_candidate': 'volume_confirmed_roll',
        'coverage': np.nan,
        'interpretability': '중간: 유동성 대표 빈티지라는 해석은 가능하나 roll rule 의존',
        'horizon_stability': roll_status,
        'liquidity_policy': '전환 규칙 자체가 거래량 조건을 포함함',
        'main_risk': roll_limit,
        'decision': '보조 후보로 보류',
    },
])

scorecard['coverage_display'] = scorecard['coverage'].map(lambda x: '-' if pd.isna(x) else f'{x:.1%}')
scorecard_display = (
    scorecard.drop(columns=['coverage'])
    .rename(columns={
        'target_candidate': '타겟 후보',
        'coverage_display': 'coverage',
        'interpretability': '해석 가능성',
        'horizon_stability': 'horizon/roll 안정성',
        'liquidity_policy': '거래량 정책',
        'main_risk': '주요 리스크',
        'decision': '현재 판단',
    })
)

scorecard_display[['타겟 후보', 'coverage', '해석 가능성', 'horizon/roll 안정성', '거래량 정책', '주요 리스크', '현재 판단']]

,타겟 후보,coverage,해석 가능성,horizon/roll 안정성,거래량 정책,주요 리스크,현재 판단
0,same_vintage_30d,90.9%,높음: 기준일과 미래일의 isu_code가 동일함,"중앙값 30일, 95% 35일","거래량≥1,000 적용 시 97.0% 유지",60d보다 정책·저빈도 변수 반영 기간이 짧을 수 있음,주력 후보
1,same_vintage_60d,82.3%,높음: 기준일과 미래일의 isu_code가 동일함,"중앙값 60일, 95% 64일","거래량≥1,000 적용 시 97.3% 유지",30d보다 coverage가 낮고 표본 말미 결측이 커짐,주력 후보
2,volume_confirmed_roll,-,중간: 유동성 대표 빈티지라는 해석은 가능하나 roll rule 의존,"후보 1개, 후보일 이후 next 우위 19.2%",전환 규칙 자체가 거래량 조건을 포함함,후보가 적고 전환 이후 다음 빈티지 우위 지속성이 약함,보조 후보로 보류


In [27]:
# 7. target schema 초안
target_schema_proposal = pd.DataFrame([
    {'column': 'trd_dd', 'dtype': 'date', 'description': '기준 거래일'},
    {'column': 'isu_code', 'dtype': 'string', 'description': '기준일 KAU 빈티지'},
    {'column': 'vwap', 'dtype': 'float', 'description': '기준일 VWAP, KRW/tCO2eq'},
    {'column': 'acc_trdvol', 'dtype': 'float', 'description': '기준일 거래량'},
    {'column': 'future_date_30d', 'dtype': 'date', 'description': '같은 빈티지 내부에서 기준일+30일 이후 첫 실제 거래일'},
    {'column': 'future_vwap_30d', 'dtype': 'float', 'description': '30d 미래일 VWAP'},
    {'column': 'actual_horizon_days_30d', 'dtype': 'float', 'description': '30d 목표 대비 실제 경과일'},
    {'column': 'target_vwap_logret_30d', 'dtype': 'float', 'description': 'log(future_vwap_30d / vwap)'},
    {'column': 'future_date_60d', 'dtype': 'date', 'description': '같은 빈티지 내부에서 기준일+60일 이후 첫 실제 거래일'},
    {'column': 'future_vwap_60d', 'dtype': 'float', 'description': '60d 미래일 VWAP'},
    {'column': 'actual_horizon_days_60d', 'dtype': 'float', 'description': '60d 목표 대비 실제 경과일'},
    {'column': 'target_vwap_logret_60d', 'dtype': 'float', 'description': 'log(future_vwap_60d / vwap)'},
    {'column': 'base_volume_ge_1000', 'dtype': 'bool', 'description': '기준일 거래량 1,000 이상 여부'},
    {'column': 'future_volume_ge_1000_30d', 'dtype': 'bool', 'description': '30d 미래일 거래량 1,000 이상 여부'},
    {'column': 'future_volume_ge_1000_60d', 'dtype': 'bool', 'description': '60d 미래일 거래량 1,000 이상 여부'},
])

target_schema_proposal

,column,dtype,description
0,trd_dd,date,기준 거래일
1,isu_code,string,기준일 KAU 빈티지
2,vwap,float,"기준일 VWAP, KRW/tCO2eq"
3,acc_trdvol,float,기준일 거래량
4,future_date_30d,date,같은 빈티지 내부에서 기준일+30일 이후 첫 실제 거래일
5,future_vwap_30d,float,30d 미래일 VWAP
6,actual_horizon_days_30d,float,30d 목표 대비 실제 경과일
7,target_vwap_logret_30d,float,log(future_vwap_30d / vwap)
8,future_date_60d,date,같은 빈티지 내부에서 기준일+60일 이후 첫 실제 거래일
9,future_vwap_60d,float,60d 미래일 VWAP


## 결론

현재 파일 기준으로는 `same_vintage_30d`와 `same_vintage_60d`를 주력 타겟 후보로 두는 것이 적절하다. `volume_confirmed_roll`은 continuous KAU 대표 시계열을 만들 수 있는 후보이지만, 02번 진단에서 전환 후보가 제한적이고 전환 이후 다음 빈티지의 거래량 우위 지속성이 약했으므로 최종 타겟으로 바로 저장하지 않는다.